# create_agent로 만드는 LangChain Agent

Agent는 모델이 필요한 Tool을 선택해 실행하고, 그 결과를 다시 읽어 최종 답변이 나올 때까지 판단을 반복하는 시스템이다. LangChain의 `create_agent()`는 이 반복 구조를 간단히 조립하는 고수준 생성 함수이며, 반환된 `CompiledStateGraph`의 실행은 LangGraph가 담당한다.

### 수동 StateGraph와 create_agent의 관계

앞 단원에서는 Agent loop의 내부 원리를 보기 위해 다음 부품을 직접 만들고 연결했다.

1. `State(messages)`: 질문·Tool 요청·Tool 결과·최종 답변을 누적한다.
2. Model Node: State를 읽어 일반 답변 또는 `tool_calls`를 만든다.
3. `ToolNode`: `tool_calls`를 실제 Python Tool 실행 결과로 바꾼다.
4. Route function: Tool 실행과 종료 중 다음 경로를 선택한다.
5. Edge와 `compile()`: `model → tools → model` 반복 경로를 실행 가능하게 만든다.

`create_agent(model, tools, ...)`는 이 표준 조립 코드를 미리 구현한다. 따라서 부품의 개념이 사라지는 것이 아니라, 필요한 설정만 전달하면 같은 Tool loop를 가진 실행 그래프가 반환된다.

```text
수동 조립: State + Model Node + ToolNode + 조건 분기 + Edge + compile()
                                  ↓ create_agent(model, tools, ...)
간편 조립:                실행 가능한 CompiledStateGraph
```

### Agent loop에서 각 설정이 놓이는 위치

- `model`: 현재 `messages`를 읽어 Tool 요청 또는 최종 답변을 만드는 판단 부품이다.
- `tools`: 모델이 선택할 수 있는 Python 기능이며 Tool 실행 결과는 다시 `messages`에 들어간다.
- `system_prompt`: 모든 모델 호출에 공통으로 적용할 고정 행동 규칙이다.
- `context_schema`와 Middleware: 호출별 `context`를 검증하고 이번 실행에 맞는 prompt를 만든다.
- `checkpointer`: `thread_id`별 State를 저장하고 다음 호출에서 복구한다.
- `response_format`: Agent loop의 최종 내용을 지정된 field와 자료형으로 검증해 `structured_response`에 저장한다.

```text
invoke(messages, context, config)
       ↓
[선택] Checkpointer가 이전 State 복구 → [선택] Middleware가 prompt 구성
       ↓
model → tool_calls → tools → ToolMessage → model
       ↓
최종 응답 생성 → [선택] response_format 검증 → 최종 State 반환
```

![LangChain 공식 Agent 구성 요소](https://cdn.jsdelivr.net/gh/goat-skn-ai/image-repo@7f008bd902003af9b0138e956af0be37ffa7a9ff/08_llm/08_langgraph/01_langgraph_basics/07_official_agent_harness.svg)

그림은 중앙의 `model`과 모델 주위를 감싸는 `harness`로 Agent를 나눈다. Harness는 모델이 사용할 Tool·지시·호출 정보·기억을 연결하는 실행 환경이다.

1. 중앙의 `model`을 먼저 본다. 이 부품이 다음 행동을 판단한다.
2. 주변의 `tools`와 `instructions`를 본다. 모델이 할 수 있는 행동과 따라야 할 규칙이다.
3. `context`와 `memory`를 본다. 현재 호출의 정보와 이전 실행의 State를 공급한다.
4. 그림의 `skills`와 `subagents`는 더 확장된 Agent의 선택 요소이며 이 실습에서는 사용하지 않는다.

> 이미지 출처: [LangChain 공식 문서 - Agents](https://docs.langchain.com/oss/python/langchain/agents)

과거의 `langgraph.prebuilt.create_react_agent` 대신 이 단원에서는 LangChain v1의 `create_agent()`를 사용한다. 반환 State의 `messages`에는 사용자 입력, Tool 요청, Tool 결과와 최종 답변이 실행 순서대로 남는다.


## LangChain Agent 패키지 설치

Agent의 조립 코드는 LangChain이 제공하고, 반환된 그래프의 반복 실행과 State 저장은 LangGraph가 담당한다. 다음 패키지는 아직 Agent를 실행하지 않고 필요한 클래스와 함수를 현재 커널에 준비한다.

- `langchain`: `create_agent()`와 Middleware 같은 고수준 Agent API를 제공한다.
- `langgraph`: Agent graph runtime과 Checkpointer를 제공한다.
- `langchain-openai`: OpenAI Chat Model을 LangChain의 `model` 부품으로 연결한다.
- `pydantic`: `response_format`에 전달할 field와 자료형을 정의한다.
- `python-dotenv`: `.env`의 환경 변수를 현재 Python 프로세스로 불러온다.


In [1]:
%pip install -U langgraph langchain langchain-openai pydantic python-dotenv


  Using cached langchain_core-1.5.6-py3-none-any.whl.metadata (4.8 kB)
Using cached langchain_core-1.5.6-py3-none-any.whl (567 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core None
Note: you may need to restart the kernel to use updated packages.


error: uninstall-no-record-file

× Cannot uninstall langchain-core None
╰─> The package's contents are unknown: no RECORD file was found for langchain-core.

hint: You might be able to recover from this via: pip install --ignore-installed --no-deps langchain-core==1.5.4


## OpenAI 인증과 선택적 LangSmith 추적 준비

OpenAI API key는 `model` 호출에 반드시 필요하지만 LangSmith는 Agent loop의 실행 과정을 관찰하는 선택 기능이다. 따라서 OpenAI key만 필수 조건으로 검사하고, LangSmith key가 있을 때만 추적 설정을 추가한다.

추적을 켜면 실행 기록은 기본적으로 `langgraph-v1-agent` 프로젝트에 모인다. `.env`나 운영체제에 `LANGSMITH_PROJECT`가 이미 설정되어 있으면 `setdefault()`가 기존 이름을 유지한다.

`find_dotenv()`가 찾은 `.env`를 `load_dotenv()`로 불러온 뒤 모델 ID를 준비한다. 이 셀의 출력은 설정 완료 문장 또는 누락된 변수 이름이며 비밀값 자체는 출력하지 않는다. 준비한 `OPENAI_CHAT_MODEL`은 다음 셀의 `ChatOpenAI` 입력으로 사용된다.


In [2]:
import os
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)

if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", "langgraph-v1-agent")

OPENAI_CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")


## model과 tools로 기본 Agent loop 조립하기

첫 실습은 `@tool → model → create_agent() → invoke() → 최종 State` 순서로 진행한다. 앞 단원에서는 State, Model Node, Tool Node, 조건 분기와 반복 Edge를 직접 연결했다.

`create_agent()`는 이 부품을 없애는 것이 아니라 내부에서 조립해 같은 Tool loop를 실행한다. 여기서 Tool loop는 `모델의 Tool 요청 → Tool 실행 → 결과를 모델에 재전달 → 최종 답변`이 나올 때까지 이어지는 반복 구조이다.

```text
get_weather 함수 --@tool--> Tool schema
OPENAI_CHAT_MODEL ---------> ChatOpenAI
Tool schema + ChatOpenAI + system_prompt ---> create_agent() ---> CompiledStateGraph
사용자 messages -----------> invoke() --------> messages가 누적된 최종 State
```

`tools`는 Agent가 실행할 수 있는 행동 목록이고, `system_prompt`는 그 행동을 언제 어떤 근거로 선택할지 안내하는 고정 규칙이다. 이 예제에서는 날씨 질문에 `get_weather` 결과만 사용하도록 정한다.

LangChain은 여러 Message 입력 형식을 허용하지만, 이 교안에서는 실제 Agent 실행의 사용자 입력을 `HumanMessage(content=...)`로 통일한다. `('human', '...')` tuple은 Prompt Template 예제에서, `{"role": "user", ...}` dict는 JSON/API 구조를 설명할 때만 사용한다.
단, 바깥의 `{"messages": [...]}` dict는 메시지 한 개를 표현하는 형식이 아니라 Agent State의 입력 구조이므로 그대로 사용한다.

Tool이 필요한 질문의 `messages`에는 보통 다음 네 기록이 순서대로 남는다.

```text
HumanMessage(질문)
→ AIMessage(tool_calls가 있는 실행 요청)
→ ToolMessage(Tool 실행 결과)
→ AIMessage(tool_calls가 없는 최종 답변)
```

코드셀 하나에서 네 단계를 함께 실행하지만, 먼저 `Tool과 model 준비`, 다음으로 `Agent loop 조립`, 마지막으로 `State 입력과 반환 messages 확인` 순서로 읽는다. 이후 예제들은 같은 loop를 유지한 채 `context와 Middleware → checkpointer → response_format` 순서로 한 부품씩 확장한다.

### `@tool`의 데이터 흐름

- 입력: 함수명, docstring, `city: str` type hint이다.
- 변환: `@tool`이 모델이 읽을 이름·설명·입력 schema가 있는 Tool 객체를 만든다.
- 출력: `get_weather`라는 Tool 객체이다.
- 다음 사용처: `create_agent(tools=[get_weather])`가 모델의 선택지와 실제 실행 함수를 함께 등록한다.

### `create_agent()`와 `invoke()`의 데이터 흐름

- 입력: 판단을 담당할 `model`, 실행 가능한 `tools`, 선택적인 `system_prompt`이다.
- 변환: Model Node·Tool Node·조건 분기·반복 Edge를 가진 실행 그래프로 조립한다.
- 출력: `invoke()`할 수 있는 `CompiledStateGraph` 계열 Agent이다.
- 다음 사용처: `invoke()`가 사용자 `messages`를 넣어 loop를 실행하고, 최종 State의 마지막 `AIMessage`를 화면에 출력한다.


In [3]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

@tool
def get_weather(city:str) -> str:
    """도시 이름이 입력되면 날씨 설명을 반환한다."""
    return f'{city}의 오늘 날씨는 매우 덥습니다.'

# LLM 모델 객체 생성
model = ChatOpenAI(model_name=OPENAI_CHAT_MODEL, use_responses_api=True)

# 에이전트 생성
# 기존 : StateGraph()를 이용해서 add_node(), add_edge, complie()
#       각 단계를 직접 작성
# create_agent(): Model Node, Tool Node, 조건 분기, 반복 Edge가
#                 포함된 실행 그래프를 자동 생성

agent = create_agent(
    model=model, # Model Node 등록
    tools=[get_weather], # Tool Node 등록 + model에게 conditional_edge 연결
    system_prompt= "날씨 질문에는 반드시 get_weather Tool의 결과만 근거로 답한다."
)

basic_response = agent.invoke(
    {

        "messages": [
            HumanMessage(content="서울 날씨는 어때?"),
        ],
    }
)
print(basic_response["messages"][-1].text)

서울의 오늘 날씨는 매우 덥습니다.


## context로 실행별 prompt를 만드는 Middleware

**Middleware**는 Agent의 기본 Tool loop를 직접 수정하지 않고, 모델을 호출하기 전이나 후에 추가 동작을 끼워 넣는 확장 기능이다.

여기서 사용하는 `@dynamic_prompt` Middleware는 모델을 호출하기 직전에 실행된다. `invoke()`에 전달된 사용자 이름을 읽어 이번 모델 호출에 사용할 `system_prompt`를 만든다.

### context는 Middleware까지 어떻게 전달되는가

- `context`: `invoke()`를 실행할 때 전달하는 호출별 정보이다. 여기서는 `UserContext` 객체에 사용자 이름을 담는다.
- `Runtime`: LangGraph가 현재 실행의 context와 설정을 담아 Middleware에 전달하는 실행 객체이다.
- `runtime.context`: Middleware가 전달된 context를 읽는 위치이다.
- `@dynamic_prompt`: `runtime.context`를 읽어 이번 모델 호출에 적용할 `system_prompt`를 반환하는 Middleware이다.

다음 호출은 대화 질문을 State의 `messages`에 넣고, 사용자 이름은 `context`에 따로 전달한다.

```python
personalized_agent.invoke(
    {
        "messages": [
            HumanMessage(content="부산 날씨를 알려 줘."),
        ],
    },
    context=UserContext(user_name="김하늘"),
)
```

처리 흐름은 다음과 같다.

```text
invoke(..., context=UserContext(user_name="김하늘"))
                         ↓
LangGraph가 context를 Runtime에 담음
                         ↓
Middleware가 request.runtime.context를 읽음
                         ↓
"사용자를 김하늘님이라고 부른다"라는 system_prompt 생성
                         ↓
이번 모델 호출에만 적용
```

따라서 `context`와 `runtime.context`는 서로 다른 정보가 아니다. 같은 정보를 호출할 때는 `context`로 입력하고, Middleware에서는 `request.runtime.context`로 읽는다. `Runtime`은 그 context를 Middleware에 전달하는 실행 객체이다.

`context`는 이번 실행의 prompt 생성에 사용되지만 `messages` State에 자동으로 추가되지는 않는다.


In [5]:
# @dataclass : 데이터를 담는 용도의 클래스
# -> __init()__, __repr__()

from dataclasses import dataclass

from langchain.agents.middleware import ModelRequest, dynamic_prompt

# UserContext : invoke(context=...)으로 전달할 호출별 정보의 자료형을 지정해둠(계약)
@dataclass
class UserContext:
    user_name: str

# 아래 함수를 모델 호출 직전에 실행되는 prompt Middleware로 등록
@dynamic_prompt
def personalized_prompt(request:ModelRequest) -> str:
    # request : 현재 모델 호출의 State, Runtime을 담은 객체

    # request.runtime.context == UserContext 객체
    user_name = request.runtime.context.user_name

    return f'날씨 Tool 결과만 근거로 답하고, 사용자를 {user_name}님이라고 부른다.'

personalized_agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[personalized_prompt],

    # invoke()의 context가 따라야 할 UserContext의 형태 (자료형)
    context_schema=UserContext
)

personalized_response = personalized_agent.invoke(
    {
        "messages": [
            HumanMessage(content="부산 날씨를 알려 줘."),
        ],
    },
    context=UserContext(user_name="김하늘"),
)

print(personalized_response["messages"][-1].text)

김하늘님, 부산은 오늘 **매우 덥습니다**.


## Checkpointer와 thread_id로 멀티턴 State 유지

Checkpointer는 그래프 실행 단계의 State를 저장하고 복구하는 객체이고, `thread_id`는 저장된 여러 대화 중 어느 State를 사용할지 선택하는 key이다. 둘 중 하나만으로는 이전 대화가 이어지지 않는다.

```text
invoke(messages, config.thread_id)
            ↓
Checkpointer가 같은 thread의 최신 messages 복구
            ↓
새 HumanMessage 병합 → Agent loop 실행
            ↓
갱신된 messages를 같은 thread의 새 Checkpoint로 저장
```

- 입력: 새 `messages`와 `configurable.thread_id`이다.
- 변환: `InMemorySaver`가 같은 thread의 이전 State를 복구하고 새 실행 결과를 다시 저장한다.
- 출력: 과거와 현재 Turn이 누적된 최종 State이다.
- 다음 사용처: 두 번째 질문의 `그럼 부산은?`이 첫 질문의 맥락을 이어받는다.

`InMemorySaver`는 현재 Python 프로세스의 RAM에만 저장하므로 커널을 종료하면 기록이 사라진다. 운영 서비스의 영구 저장소가 아니라 Checkpoint 동작을 확인하기 위한 수업용 구현이다.


In [8]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model=model,
    tools=[get_weather],

    # 실행 단계의 State를 Thread별 checkpoint로 메모리에 저장하는 객체
    checkpointer = InMemorySaver(),
)

memory_agent.invoke(
    {
        "messages": [ HumanMessage(content="서울 날씨 알려줘")]
    },
    config = {
        "configurable" : { "thread_id": "weather-thread-1"}
    }
)

{'messages': [HumanMessage(content='서울 날씨 알려줘', additional_kwargs={}, response_metadata={}, id='5b3bc35d-1687-450e-b236-600ec7c71d41'),
  AIMessage(content=[{'arguments': '{"city":"서울"}', 'call_id': 'call_S8YTJCcztzGFQ7SiApeBX0wP', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0f852f20c8f39d01006a850653654487d089c26275845c6dd0', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0f852f20c8f39d01006a850651f21087d0a2398b3aa1bdc8e6', 'created_at': 1787102802.0, 'metadata': {}, 'model': 'gpt-5.6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna'}, id='resp_0f852f20c8f39d01006a850651f21087d0a2398b3aa1bdc8e6', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_S8YTJCcztzGFQ7SiApeBX0wP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 18, 'total_tokens': 72, 'input_token_details': {'cache_creation': 

In [9]:
follow_up = memory_agent.invoke(
    {
        "messages": [ HumanMessage(content="방금 물어본 지역이 어디라고?")]
    },
    config = {
        "configurable" : { "thread_id": "weather-thread-1"}
    }
)

print(follow_up["messages"][-1].text)

{'messages': [HumanMessage(content='서울 날씨 알려줘', additional_kwargs={}, response_metadata={}, id='5b3bc35d-1687-450e-b236-600ec7c71d41'),
  AIMessage(content=[{'arguments': '{"city":"서울"}', 'call_id': 'call_S8YTJCcztzGFQ7SiApeBX0wP', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0f852f20c8f39d01006a850653654487d089c26275845c6dd0', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0f852f20c8f39d01006a850651f21087d0a2398b3aa1bdc8e6', 'created_at': 1787102802.0, 'metadata': {}, 'model': 'gpt-5.6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna'}, id='resp_0f852f20c8f39d01006a850651f21087d0a2398b3aa1bdc8e6', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_S8YTJCcztzGFQ7SiApeBX0wP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 18, 'total_tokens': 72, 'input_token_details': {'cache_creation': 

## response_format으로 최종 결과 구조화하기

일반 `messages`의 마지막 답변은 사람이 읽기 쉬운 자유 형식 문장이다. 후속 프로그램이 도시명과 날씨 상태를 안정적으로 사용해야 한다면 `response_format`에 Pydantic schema를 전달해 최종 결과의 field와 자료형을 정할 수 있다.

```text
사용자 messages → Agent Tool loop가 Tool 호출을 마침
                                      ↓
                  response_format 전략으로 최종 응답 생성
                                      ↓
                         Pydantic field·자료형 검증
                                      ↓
                      State["structured_response"]
```

- 입력: `WeatherResponse` schema type과 사용자 `messages`이다.
- 변환: LangChain이 모델 기능에 맞는 구조화 출력 전략을 선택하고 Pydantic이 결과를 검증한다.
- 출력: `WeatherResponse` 객체가 State의 `structured_response` key에 저장된다.
- 다음 사용처: 후속 Python 코드가 `city`와 `conditions`를 문자열 분석 없이 field로 읽을 수 있다.

구조화 출력은 형식과 자료형을 검증하지만 Tool 반환값의 사실성까지 보장하지는 않는다. `messages`와 `structured_response`는 같은 State에 있지만 목적이 다른 결과이다.


In [10]:
from pydantic import BaseModel, Field

class WeatherResponse(BaseModel):
    city: str = Field(description="Tool이 조회한 도시 이름")
    conditions: str = Field(description="Tool 결과에 근거한 날씨 상태")

structured_agent = create_agent(
    model=model,
    tools=[get_weather],
    response_format=WeatherResponse,
)

structured_result = structured_agent.invoke(
    {
        "messages": [
            HumanMessage(content="인천 날씨를 알려 줘."),
        ],
    }
)
# structured_response: schema 검증을 통과한 WeatherResponse 객체
print(structured_result["structured_response"])

city='인천' conditions='오늘 날씨는 매우 덥습니다.'


## 요구사항에 맞는 create_agent 확장 선택

`create_agent()`가 표준 loop 조립을 대신하므로, 실제 구현에서는 문제에 필요한 부품만 인자로 추가한다.

- 기본 Tool loop: `model`과 `tools`로 판단과 실행 기능을 연결한다.
- 고정 정책: `system_prompt`로 모든 모델 호출에 같은 행동 규칙을 적용한다.
- 호출별 정보: `context_schema`와 Middleware를 등록하고 `invoke(context=...)`로 값을 주입한다.
- 대화 State 복구: `checkpointer`를 연결하고 `invoke(config=...)`에 `thread_id`를 전달한다.
- 프로그램용 최종 결과: `response_format`으로 `structured_response`의 schema를 정한다.

```text
필수 부품              model + tools
입력 지침 확장          system_prompt 또는 Middleware + context
실행 사이 State 유지    checkpointer + thread_id
최종 출력 계약          response_format
```

`context`는 한 번의 실행에 주입하는 읽기 전용 정보이고, Checkpoint는 Agent가 실행하며 갱신한 State의 snapshot이다. `response_format`은 저장 방식을 바꾸는 기능이 아니라 최종 결과의 field 구조를 정한다. 가장 단순한 Agent에서 시작해 실제 요구사항이 생길 때 필요한 부품만 추가한다.
